# Seleção de ofertas em campanhas de marketing direto

Implementação dos modelos M1/M2 e do set covering M3 do problema descrito por Talla Nobibon, Leus e Spieksma (2011). O código reutilizável está nos módulos deste projeto; execute o notebook a partir da pasta do repositório, após instalar as dependências com `python -m pip install -e ".[test]"`.

O exemplo reproduz os valores especificados, resolve as formulações com HiGHS, compara as relaxações lineares e confere o ótimo por enumeração independente. O gerador segue a Secção 5.1, com o teto de MQC ajustado para `floor(2 ΣM_i/n)` para evitar intervalos impossíveis de orçamento; a variante aleatória de orçamento usa o limite superior corrigido descrito em `generator.py`. Custos fixos fora de um intervalo inteiro admissível geram aviso e usam `max(0, ceil(limite inferior))`.

In [1]:
from instance import validation_instance
from models import brute_force_optimum, solve_model
from setcover import enumerate_columns, solve_setcover

instance = validation_instance()
print(f"Clientes: {instance.m}; produtos: {instance.n}; R: {instance.hurdle_rate:.2f}")

Clientes: 3; produtos: 2; R: 0.10


In [2]:
columns = enumerate_columns(instance)
print(f"Colunas admissíveis: {len(columns)} de 14 conjuntos candidatos não vazios")
for column in columns:
    product = instance.product_names[column.product]
    customers = tuple(i + 1 for i in column.customers)
    print(f"{product} {customers}: lucro={column.objective_coefficient:g}, hurdle={column.hurdle_coefficient:.1f}")

Colunas admissíveis: 8 de 14 conjuntos candidatos não vazios
A (1, 2): lucro=9, hurdle=8.5
A (1, 3): lucro=10, hurdle=9.4
B (1,): lucro=2, hurdle=1.8
B (2,): lucro=1, hurdle=0.8
B (3,): lucro=1, hurdle=0.7
B (1, 2): lucro=4, hurdle=3.7
B (1, 3): lucro=4, hurdle=3.6
B (2, 3): lucro=3, hurdle=2.6


In [3]:
brute_value, brute_x, brute_y, optimum_count = brute_force_optimum(instance)
print(f"Ótimo por força bruta: {brute_value:g}; soluções ótimas: {optimum_count}")
print("x (linhas=clientes, colunas=A/B):")
for row in brute_x:
    print(row)
print("y:", brute_y)
assert brute_value == 11 and optimum_count == 1

Ótimo por força bruta: 11; soluções ótimas: 1
x (linhas=clientes, colunas=A/B):
(1.0, 0.0)
(0.0, 1.0)
(1.0, 0.0)
y: (1.0, 1.0)


In [4]:
integer_results = {
    "M1": solve_model(instance, "M1"),
    "M2": solve_model(instance, "M2"),
    "M3": solve_setcover(instance),
}
for name, result in integer_results.items():
    print(f"{name}: objetivo={result.objective:g}; estado={result.termination}; tempo={result.elapsed_seconds:.3f}s")
    assert result.objective == 11

M1: objetivo=11; estado=optimal; tempo=0.026s
M2: objetivo=11; estado=optimal; tempo=0.026s
M3: objetivo=11; estado=optimal; tempo=0.032s


In [5]:
lp_values = {
    "M1": solve_model(instance, "M1", relax=True).objective,
    "M2": solve_model(instance, "M2", relax=True).objective,
    "M3": solve_setcover(instance, relax=True).objective,
}
print("Limites LP:", ", ".join(f"{name}={value:.6f}" for name, value in lp_values.items()))
assert 11 <= lp_values["M3"] + 1e-7 <= lp_values["M2"] + 1e-7 <= lp_values["M1"] + 1e-7

Limites LP: M1=12.888889, M2=12.666667, M3=11.000000


In [6]:
from generator import generate_budget_variants

small_instances = generate_budget_variants(
    m=100, n=5, hurdle_rate=0.10, capacity_variant="s", seed=2026
)
for budget_type, generated in small_instances.items():
    print(
        f"{budget_type}: {generated.m} clientes, {generated.n} produtos, "
        f"orçamentos={generated.budgets}, custos fixos={generated.fixed_costs}"
    )

# Geração do plano experimental (162 instâncias por padrão):
# python generator.py --output-dir instances --seed 1
# Resolução M1/M2 e relatórios CSV:
# python run_experiments.py --instances-dir instances

minimum: 100 clientes, 5 produtos, orçamentos=(64.0, 46.0, 81.0, 65.0, 85.0), custos fixos=(147.0, 93.0, 158.0, 116.0, 168.0)
maximum: 100 clientes, 5 produtos, orçamentos=(81.0, 77.0, 84.0, 87.0, 86.0), custos fixos=(147.0, 93.0, 158.0, 116.0, 168.0)
random: 100 clientes, 5 produtos, orçamentos=(81.0, 71.0, 82.0, 84.0, 86.0), custos fixos=(147.0, 93.0, 158.0, 116.0, 168.0)


## Experimento S1: comparação entre solvers

As próximas células geram as 54 instâncias S1 (m=100: 3 valores de n × 3 taxas R × 2 variantes de M × 3 orçamentos), resolvem M1 e M2 com HiGHS, Gurobi e CPLEX e salvam resultados detalhados em `results/S1_results.csv` e agregados em `results/S1_aggregate.csv`. Para cada combinação solver/instância/modelo são executadas a resolução inteira e a relaxação LP. A duração depende do equipamento e das licenças; configure `TIME_LIMIT_SECONDS` antes de iniciar. Nenhum solver indisponível será ignorado silenciosamente.

In [ ]:
from pathlib import Path
from generator import generate_budget_variants

SEED = 1
S1_DIR = Path("instances") / "S1"
S1_DIR.mkdir(parents=True, exist_ok=True)
S1_INSTANCE_PATHS = []

for n in (5, 10, 15):
    for hurdle_rate in (0.05, 0.10, 0.15):
        for capacity_variant in ("s", "l"):
            variants = generate_budget_variants(
                m=100,
                n=n,
                hurdle_rate=hurdle_rate,
                capacity_variant=capacity_variant,
                seed=SEED,
            )
            for budget_variant, generated in variants.items():
                path = S1_DIR / (
                    f"S1_n{n}_R{hurdle_rate * 100:.0f}_"
                    f"{capacity_variant}_{budget_variant}_seed{SEED}.json"
                )
                generated.to_json(path)
                S1_INSTANCE_PATHS.append(path)

assert len(S1_INSTANCE_PATHS) == 54
print(f"S1: {len(S1_INSTANCE_PATHS)} instâncias geradas em {S1_DIR.resolve()}")

In [ ]:
import pyomo.environ as pyo

S1_SOLVERS = ("appsi_highs", "gurobi", "cplex")
unavailable = [
    solver
    for solver in S1_SOLVERS
    if not pyo.SolverFactory(solver).available(exception_flag=False)
]
if unavailable:
    raise RuntimeError(
        "Configure os seguintes solvers antes de executar o experimento: "
        + ", ".join(unavailable)
    )
print("Solvers disponíveis para o S1:", ", ".join(S1_SOLVERS))

In [ ]:
from run_experiments import (
    run_experiments,
    write_aggregate_report,
    write_results_csv,
)

TIME_LIMIT_SECONDS = 3600
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
S1_RESULTS = []

for solver_name in S1_SOLVERS:
    print(f"Executando S1 com {solver_name}...")
    solver_rows = run_experiments(
        S1_INSTANCE_PATHS,
        RESULTS_DIR / f"S1_{solver_name}.csv",
        solver=solver_name,
        time_limit=TIME_LIMIT_SECONDS,
    )
    S1_RESULTS.extend(solver_rows)

S1_RESULTS_PATH = RESULTS_DIR / "S1_results.csv"
S1_AGGREGATE_PATH = RESULTS_DIR / "S1_aggregate.csv"
write_results_csv(S1_RESULTS, S1_RESULTS_PATH)
S1_AGGREGATE = write_aggregate_report(S1_RESULTS, S1_AGGREGATE_PATH)
print(
    f"Concluído: {len(S1_RESULTS)} linhas detalhadas; "
    f"arquivos consolidados em {RESULTS_DIR.resolve()}"
)

In [ ]:
from collections import defaultdict
from statistics import mean, median

if not S1_RESULTS:
    raise RuntimeError("Execute primeiro a célula 'run-s1-solvers'.")

analysis_groups = defaultdict(list)
for row in S1_RESULTS:
    analysis_groups[(row["solver"], row["model"])].append(row)

print("Resumo geral por solver e modelo")
print(f"{'Solver':<14} {'Modelo':<7} {'Ótimos':>12} {'Gap LP médio':>15} {'Tempo mediano (s)':>20} {'Gap MIP médio':>15}")
for (solver_name, model_name), rows in sorted(analysis_groups.items()):
    optimal_count = sum(row["termination"].lower() == "optimal" for row in rows)
    lp_gaps = [float(row["lp_gap_percent"]) for row in rows if row["lp_gap_percent"] is not None]
    mip_gaps = [float(row["mip_gap_percent"]) for row in rows if row["mip_gap_percent"] is not None]
    times = [float(row["elapsed_seconds"]) for row in rows]
    print(
        f"{solver_name:<14} {model_name:<7} {optimal_count:>5}/{len(rows):<6} "
        f"{mean(lp_gaps) if lp_gaps else float('nan'):>14.3f}% "
        f"{median(times):>20.3f} "
        f"{mean(mip_gaps) if mip_gaps else float('nan'):>14.4f}%"
    )

print("\nDetalhe por tamanho de n (18 instâncias por n e por solver/modelo)")
print(f"{'Solver':<14} {'Modelo':<7} {'n':>3} {'Ótimos':>10} {'Gap LP médio':>15} {'Tempo médio (s)':>17}")
for key in sorted({(row['solver'], row['model'], row['n']) for row in S1_RESULTS}):
    solver_name, model_name, n_value = key
    rows = [row for row in S1_RESULTS if (row['solver'], row['model'], row['n']) == key]
    optimal_count = sum(row["termination"].lower() == "optimal" for row in rows)
    lp_gaps = [float(row["lp_gap_percent"]) for row in rows if row["lp_gap_percent"] is not None]
    times = [float(row["elapsed_seconds"]) for row in rows]
    print(
        f"{solver_name:<14} {model_name:<7} {n_value:>3} "
        f"{optimal_count:>4}/{len(rows):<5} "
        f"{mean(lp_gaps) if lp_gaps else float('nan'):>14.3f}% "
        f"{mean(times):>17.3f}"
    )

print("\nInterpretação: compare tempos e proporção de ótimos provados para avaliar desempenho do solver.")
print("Os gaps LP medem a força da formulação (M1/M2), não uma vantagem do solver; devem coincidir entre solvers, salvo tolerâncias numéricas.")
print("O gap MIP compara incumbent e bound quando a otimalidade não foi provada. Consulte os CSVs para estados por instância.")
print(f"Relatórios: {S1_RESULTS_PATH.resolve()} e {S1_AGGREGATE_PATH.resolve()}")